## DS 3100 Statistics Refresher — Companion Notebook (Python)

This notebook accompanies the inferential statistics refresher. We focus on **correlation as the statistic of interest**. Work through the questions in order: first interpret the observed correlation, then connect it to the null hypothesis, sampling variation, p-values, effect size, and power.

Use the notebook to distinguish clearly between:
- the **population parameter** ($\rho$)
- the **sample statistic** ($r$)
- the **test statistic** used for inference
- the **sampling distribution**
- the **$p$-value**

### Setup

Let's make some fictional clinic waiting-time data with two variables:

- `wait_min`: patient waiting time in minutes
- `satisfaction`: patient satisfaction score

and model a realistic negative relationship between the two.

For this activity, treat the data frame as a sample from a larger population of patients.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, t, nct

rng = np.random.default_rng(3100)

n = 120
wait_min = np.round(rng.uniform(5, 90, n), 1)
satisfaction = np.round(85 - 0.35 * wait_min + rng.normal(0, 10, n), 1)

clinic = pd.DataFrame({
    "wait_min": wait_min,
    "satisfaction": satisfaction
})

n_obs = len(clinic)

clinic.head()

### 1. Start with the observed sample

Create a scatterplot of `wait_min` against `satisfaction`. Then calculate Pearson's correlation.

**Question:** What does the sign of the observed correlation tell you? What does its magnitude tell you?

In [ ]:
# Scatterplot using matplotlib
plt.scatter(
    ,
    
)
plt.xlabel("Waiting Time (minutes)")
plt.ylabel("Satisfaction")
plt.title("Waiting Time and Patient Satisfaction")
plt.show()

In [ ]:
# Calculate Pearson's r using scipy.stats.pearsonr
result = pearsonr(
    ,
    
)

r_obs = result.statistic
r_obs

### 2. Identify the objects necessary for inference

Suppose your observed correlation is called `r_obs`. Complete the following conceptual mapping before we proceed further.

- `r_obs` = ?
- $\rho$ = ?
- Is `r_obs` fixed or could it change with a new random sample? Explain briefly.

**Solution**

- `r_obs` ...
- $\rho$ = ...
- `r_obs` ...

### 3. State the hypotheses

Our research question:

> **Are patient waiting time and satisfaction associated in the population?**

Begin by writing the null and alternative hypotheses. Use a **two-tailed** alternative first.



$$
H_0: ...
$$

$$
H_A: ...
$$


### 4. Simulate the null sampling distribution

Now assume the null hypothesis is true: the population correlation ($\rho$) is zero.

To understand the $p$-value, we need to imagine what sample correlations would look like **if this null hypothesis were true**.

> **Important:** The **null setting** is hypothetical; it is not an actual claim that `wait_min` and `satisfaction` are actually independent in the clinic population.

For this simulation, we create two independent variables so that their population correlation is $0$. We then:

1. draw a sample of $n=n_{obs}$ observations;
2. calculate Pearson's $r$;
3. repeat this **5,000 times**;
4. plot the resulting **sampling distribution of $r$ under $H_0$**.

This distribution represents the values of $r$ we could obtain through sampling variation **when there is no population correlation**.

In [ ]:
# Simulate 5,000 correlations under H_0 with the observed sample size
rng_null = np.random.default_rng(3100)

null_r = []

for _ in range(5000):
    x = rng_null.normal(
        loc=0,
        scale=1,
        size=n_obs
    )
    y = rng_null.normal(
        loc=0,
        scale=1,
        size=n_obs
    )

    null_r.append(np.corrcoef(x, y)[0, 1])

null_r = np.array(null_r)

In [ ]:
# Histogram of the null sampling distribution
plt.hist(
    null_r,
    bins=30,
    edgecolor="black"
)
plt.xlabel("Correlation (r)")
plt.ylabel("Frequency")
plt.title("Sampling Distribution of r Under H0")
plt.show()

### 5. Put the observed correlation on the null distribution

Use the sample correlation statistic and add a vertical line at the observed value. For a two-tailed test, also consider the equally extreme value on the opposite side.

**Question:** Looking at this simulated null distribution, how extreme does the observed correlation look?

In [ ]:
plt.hist(
    null_r,
    bins=30,
    edgecolor="black"
)

# Observed correlation
plt.axvline(
    r_obs,
    linewidth=2
)

# Equally extreme value in the opposite tail
plt.axvline(
    -r_obs,
    linewidth=2,
    linestyle="--"
)

plt.xlabel("Correlation (r)")
plt.ylabel("Frequency")
plt.title("Sampling Distribution of r Under H0")
plt.show()

### 6. Calculate the p-value with a standard package

Now perform the Pearson correlation test using `scipy.stats.pearsonr()`. This function does more than simply calculate $r$. Use it to identify the following:

- the observed sample correlation `r`;
- the **test statistic**, a standardized version used for inference;
- the **degrees of freedom**, here $df=n-2$ for the Pearson correlation test;
- the **$p$-value**.
- set the `alternative` parameter for a two sided test.

In [ ]:
two_tailed_result = pearsonr(
    ,
    ,
    alternative=
)

r_obs = two_tailed_result.statistic
test_statistic = (
    r_obs * np.sqrt((n_obs - 2) / (1 - r_obs**2))
)
df = n_obs - 2
p_value = two_tailed_result.pvalue

print("Observed r:", round(r_obs, 3))
print("Test statistic:", round(test_statistic, 3))
print("Degrees of freedom:", df)
print("p-value:", f"{p_value:.4g}")

### 7. One-tailed versus two-tailed

Change the research question to:

> **Does longer waiting time correspond to lower satisfaction?**

State the new alternative hypothesis and run the appropriate one-tailed correlation test by changing the `alternative` parameter. Compare its $p$-value with the two-tailed result.

In [ ]:
# H0: rho = 0
# HA: rho < 0 (longer waiting time is associated with lower satisfaction)

one_tailed_result = pearsonr(
    ,
    ,
    alternative=
)

print("Two-tailed p-value:", f"{two_tailed_result.pvalue:.4g}")
print("One-tailed p-value:", f"{one_tailed_result.pvalue:.4g}")

#### Do you think based on these p-values there is strong statistical evidence for a negative correlation at a conventional significance level of 0.05?

### 8. Effect size versus significance

The sample correlation `r` is itself an effect-size measure for linear association.

Imagine two studies both estimate approximately `r = -0.20`, but one uses a small sample and one uses a very large sample.

**Question:** Why can the $p$-values differ even though the estimated effect is the same?

### 9. Power

Suppose a population correlation of approximately $\rho=-0.20$ really exists.

We will use a standard power-analysis calculation to investigate how the probability of detecting this effect changes with sample size. Compare at least `n = 25`, `50`, `100`, and `200` with significance level 0.05. The `pingouin` package has a function specifically for correlation power `pingouin.power_corr()`.

**Question:** What happens to power as sample size increases? Explain intuitively.

In [ ]:
import pingouin as pg

sample_sizes = [25, 50, 100, 200]

for n in sample_sizes:
    power = pg.power_corr(
        r= ,
        n= ,
        alpha= ,
        alternative= 
    )

    print(
        f"n = {n} | power = {power:.3f}"
    )

### 10. Putting everything together

Reiterate our chain of thought for the waiting-time/satisfaction example by explaining each of the following:

`Research question` → `population parameter` → `sample statistic` → `null hypothesis` → `test statistic` → `reference distribution` → `$p$-value` → `effect size` → `interpretation`

Write a final 3–4 sentence interpretation that distinguishes **association from causation** and **statistical significance from practical importance**.

- **Research question**: ...
- **Population parameter**: ...
- **Sample statistic**: ...
- **Null hypothesis**: ...
- **Test statistic**: ...
- **Reference distribution**: ...
- **$p$-value**: ...
- **Effect size**: ...
- **Interpretation**: ...

...
